# 3. One report becomes three

You can explain one report. Now a colleague asks for the same report on three stocks.
The new idea is small: **call the same operation in a loop, changing only its inputs**.

This notebook runs independently; you do not need to run the report template manually first.
We request AAPL, MSFT, and NVDA, plus **one intentionally invalid ticker**. Three reports should succeed and one should be logged as failed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')
import papermill as pm
import scrapbook as sb
import os
import sys
import tempfile
import json
from contextlib import closing

## 1. Trace one run
Papermill opens our template, inserts parameter values after its tagged `parameters` cell, runs it in a fresh Python session, and saves the executed notebook.
The template's variables do not depend on another notebook's memory.

The next cell chooses the database and output folder and configures reports to use the current Python environment.

In [ ]:
template = lesson_dir / '2_stock_analysis_template.ipynb'
db_path = lesson_dir / 'data/data.db'
if not db_path.is_file():
    raise FileNotFoundError(f'Download the bCourses database to {db_path}')
conn = sqlite3.connect(db_path.resolve().as_uri() + '?mode=ro', uri=True)
available = pd.read_sql_query('SELECT DISTINCT ticker FROM ohlc ORDER BY ticker', conn)['ticker'].tolist()
conn.close()
print('Available tickers:', available)
tickers = ['AAPL', 'MSFT', 'NOT_A_TICKER', 'NVDA']  # One deliberately invalid request.
start_date, end_date = '2024-01-01', '2024-06-30'
# Supplied helper: create a new output folder and use this Python interpreter for child reports.
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
from report_runtime import prepare_report_run
run_dir, kernel_name = prepare_report_run(lesson_dir / 'outputs')
print('Reports will be saved in:', run_dir)

Read `run_report` as a reusable recipe. Its inputs are ticker, start, end, and destination folder. It returns the saved report's path.
**Predict:** if only ticker changes, which report sections will change? Will another stock inherit AAPL's variables?

In [ ]:
def run_report(ticker, start, end, folder):
    """Run one independent report; propagate errors so the caller can record failures."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output = folder / f'{ticker}_{start}_{end}.ipynb'
    pm.execute_notebook(
        str(template), str(output),
        parameters={'ticker': ticker, 'start_date': start, 'end_date': end,
                    'db_path': str(db_path.resolve()), 'volatility_window': 20},
        kernel_name=kernel_name, cwd=str(lesson_dir), progress_bar=False,
    )
    return output

first_report = run_report(tickers[0], start_date, end_date, run_dir)
first = sb.read_notebook(str(first_report))
print('First report:', first.scraps['ticker'].data, first.scraps['total_return_pct'].data)
print('Scraps:', list(first.scraps))

## 2. A loop and one deliberate failure
Before running, write down the expected number of successful reports and failures.
The loop starts after the already completed AAPL report. `try` attempts a report; `except` records an error so later tickers can still run.
A failed notebook can exist on disk. **Existence is not evidence of a successful calculation.**

In [ ]:
successful_paths = [first_report]
failures = []
for symbol in tickers[1:]:
    try:
        successful_paths.append(run_report(symbol, start_date, end_date, run_dir))
    except Exception as exc:
        failures.append({'ticker': symbol, 'error': str(exc)})
        print(f'FAILED: {symbol}. The full error is saved in failed_stocks.json.')
(run_dir / 'failed_stocks.json').write_text(json.dumps(failures, indent=2))
(run_dir / 'manifest.json').write_text(json.dumps({
    'start_date': start_date, 'end_date': end_date,
    'reports': [path.name for path in successful_paths], 'failures': failures}, indent=2))
print(f'Successfully generated {len(successful_paths)}/{len(tickers)} reports')

### Retrieve results, do not scrape printed tables
The template used `sb.glue` to record named results. `sb.read_notebook(...).scraps` reads them back.
The provided collector accepts only paths from successful runs. It refuses duplicate report keys and an empty result list.
**Discuss:** did NVDA still run after the invalid ticker? Is the invalid ticker absent from the summary?

In [ ]:
def collect_reports(paths):
    """Build a summary from successful paths only; reject empty or duplicated input."""
    rows = []
    names = ['ticker', 'start_date', 'end_date', 'trading_days', 'total_return_pct',
             'avg_volatility', 'max_drawdown_pct', 'data_quality_ok']
    for path in paths:
        scraps = sb.read_notebook(str(path)).scraps
        row = {name: scraps[name].data for name in names}
        row['report_path'] = str(path)
        rows.append(row)
    if not rows:
        raise ValueError('No successful reports to summarize; inspect failed_stocks.json.')
    summary = pd.DataFrame(rows)
    if summary.duplicated(['ticker', 'start_date', 'end_date']).any():
        raise ValueError('Duplicate report keys in this run.')
    return summary.sort_values('total_return_pct', ascending=False).reset_index(drop=True)

summary = collect_reports(successful_paths)
assert summary['start_date'].eq(start_date).all() and summary['end_date'].eq(end_date).all()
assert summary['data_quality_ok'].all()
summary.to_csv(run_dir / 'portfolio_summary.csv', index=False)
display(summary.drop(columns='report_path'))

## 3. Worked example: compare the deepest drawdowns
Select ticker, observation count, and maximum drawdown. Drawdown is stored as a negative percentage here, so ascending order puts the deepest loss first.

In [ ]:
example_drawdown_comparison = summary[['ticker', 'trading_days', 'max_drawdown_pct']].sort_values('max_drawdown_pct')
display(example_drawdown_comparison)
print('Deepest drawdown in this period:', example_drawdown_comparison.iloc[0]['ticker'])
print('The comparison describes these stocks over the same historical period.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
summary.sort_values('total_return_pct').plot.barh(x='ticker', y='total_return_pct',
    ax=axes[0], legend=False, title='Total return over the common period', xlabel='Return (%)')
axes[1].scatter(summary['avg_volatility'] * 100, summary['total_return_pct'])
for row in summary.itertuples():
    axes[1].annotate(row.ticker, (row.avg_volatility * 100, row.total_return_pct),
                     xytext=(4, 4), textcoords='offset points')
axes[1].margins(x=0.15, y=0.15)
axes[1].set(xlabel='Mean rolling volatility, annualized (%)', ylabel='Total return (%)',
            title='Return and variability')
fig.savefig(run_dir / 'comparison.png', dpi=150, bbox_inches='tight')
plt.show()
# Full data also travels with the report, if you need to audit a metric.
aapl_prices = first.scraps['prices'].data
assert np.isclose(first.scraps['total_return_pct'].data,
                  (aapl_prices['adj_close'].iloc[-1] / aapl_prices['adj_close'].iloc[0] - 1) * 100)
display(aapl_prices.tail(3))

## 4. Exit explanation and homework map
Without looking at the code, describe: **parameters → one report → loop → successful outputs → summary**.
Explain why a clean-looking output folder can still contain unusable results.

[Homework 2](../Lecture%202/hw2.md) is due October 29. The deliverables build on these patterns:

| Deliverable | Pattern you now have |
|---|---|
| Clean `ohlc_hw`, document and validate | October 8 masks/repairs + today's SQL and quality checks |
| Parameterized risk report | Functions, tiny-data checks, rolling statistics, Figure/Axes, and the tagged parameter cell |
| Reports for all stocks and a summary | This loop, recorded failures, and named outputs |

Homework adds its specified metrics and styling, including Sharpe ratio, and expands the three-stock demonstration to all cleaned tickers.
For six complete months, use the latest **database** date as the anchor and exclude its unfinished month; a computer's current date would be inappropriate for this historical data.
Later advanced-Python and production lessons build on this workflow with program structure and testing.

## Optional extension: multiple periods
Set `RUN_EXTENSIONS = True` to run three valid tickers over all four 2023 quarters (12 additional reports).
The data coverage for these stocks is sufficient in 2023; Q4 2024 has only 14 observations and cannot support the 20-return window.
The nested loop is the same recipe with one additional changing input.

In [ ]:
RUN_EXTENSIONS = False
periods = [('2023-01-01', '2023-03-31', 'Q1'), ('2023-04-01', '2023-06-30', 'Q2'),
           ('2023-07-01', '2023-09-30', 'Q3'), ('2023-10-01', '2023-12-31', 'Q4')]
if RUN_EXTENSIONS:
    quarterly_paths, quarterly_failures = [], []
    for symbol in ['AAPL', 'MSFT', 'NVDA']:
        for start, end, label in periods:
            try:
                quarterly_paths.append(run_report(symbol, start, end, run_dir / 'quarterly'))
            except Exception as exc:
                quarterly_failures.append({'ticker': symbol, 'period': label, 'error': str(exc)})
    (run_dir / 'quarterly_failures.json').write_text(json.dumps(quarterly_failures, indent=2))
    quarterly = collect_reports(quarterly_paths)
    labels = {start: label for start, end, label in periods}
    quarterly['period'] = quarterly['start_date'].map(labels)
    display(quarterly.pivot(index='ticker', columns='period', values='total_return_pct'))
    display(quarterly.pivot(index='ticker', columns='period', values='avg_volatility'))
else:
    print('Set RUN_EXTENSIONS = True to run the quarterly reports.')

The temporary kernel definition is inside this run's output folder and is active only in this notebook process. It is not installed into your user settings.
You may call `run_report` again in this session; restarting the kernel and running the setup creates a fresh output folder.

In [ ]:
print('Reports, failure log, manifest, comparison figure, and summary CSV:', run_dir)

## Exercises
Use the tables and functions above to complete these tasks. Replace the placeholders with your code, then run the checks.

### Your turn H: answer a new desk question
Create a table called `your_comparison` with **ticker, total_return_pct, and avg_volatility**, sorted by average volatility from largest to smallest.
Use only the successful summary rows. Write `your_comparison_note` naming the most variable stock **in this period** and one limitation of the comparison.

**Hint:** column selection followed by `.sort_values('avg_volatility', ascending=False)`.
**Stretch:** add a percent-formatted volatility column without changing the stored fractional metric.
Examine the chart and explain how it relates to your sorted table.

In [ ]:
your_comparison = None
your_comparison_note = ''

In [ ]:
if your_comparison is None:
    print('Try H: select three columns, sort, and explain the result in its time period.')
else:
    comparison_expected = summary[['ticker', 'total_return_pct', 'avg_volatility']].sort_values('avg_volatility', ascending=False)
    pd.testing.assert_frame_equal(your_comparison.reset_index(drop=True), comparison_expected.reset_index(drop=True))
    assert 'NOT_A_TICKER' not in set(your_comparison['ticker'])
    assert your_comparison_note.strip()
    print('H passed. The descriptive comparison alone is not an investment recommendation.')